In [1]:
import numpy as np
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X_1 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data\feature\train\lfcc_lbp_speech_80_nodelta_aurora.npy")
X_2 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data\feature\train\lfcc_lbp_speech-noise_80_nodelta_aurora.npy")

X = np.vstack([X_1, X_2]).astype(np.float32)
y = np.hstack([
    np.ones(len(X_1), dtype=np.int8),
    np.ones(len(X_2), dtype=np.int8),
])

X, y = shuffle(X, y, random_state=42)

print("Final shape:", X.shape, y.shape)



# # ===== giảm tiếp xuống 3D để vẽ =====
# pca_3d = PCA(n_components=3)
# X_3d = pca_3d.fit_transform(X)


# # ===== Vẽ 3D =====
# fig = plt.figure(figsize=(10,8))
# ax = fig.add_subplot(111, projection='3d')

# ax.scatter(X_3d[y==0,0], X_3d[y==0,1], X_3d[y==0,2], label="bee", alpha=0.6, s=8)
# ax.scatter(X_3d[y==1,0], X_3d[y==1,1], X_3d[y==1,2], label="nobee", alpha=0.6, s=8)
# ax.scatter(X_3d[y==2,0], X_3d[y==2,1], X_3d[y==2,2], label="noqueen", alpha=0.6, s=8)

# ax.set_title(f"3D PCA (retain {variance_keep*100:.0f}% variance)")
# ax.set_xlabel("PC1")
# ax.set_ylabel("PC2")
# ax.set_zlabel("PC3")

# ax.legend()
# plt.show()

Final shape: (956, 256) (956,)


In [2]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Hàm tính EER =====
def compute_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer

spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

scale_pos_weight = spoof / bonafide

print("scale_pos_weight:", scale_pos_weight)


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
precisions = []
recalls = []
f1s = []
aucs = []
eers = []

count = 1

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_score))
    eers.append(compute_eer(y_test, y_score))

    # print(f"Accuracy: {accs[-1]}")
    # print(f"Precision: {precisions[-1]}")
    # print(f"Recall: {recalls[-1]}")
    # print(f"F1: {f1s[-1]}")
    # print(f"EER: {eers[-1]}")
    # print()


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

scale_pos_weight: 0.0
Training phase: 1


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0], got [1]

In [4]:
from sklearn.ensemble import RandomForestClassifier

spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

print("spoof:", spoof)
print("bonafide:", bonafide)


# ===== 5 Fold CV =====
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs = []
precisions = []
recalls = []
f1s = []
aucs = []
eers = []

count = 1

for train_idx, test_idx in kf.split(X, y):

    print(f"Training phase: {count}")
    count += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    accs.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))
    aucs.append(roc_auc_score(y_test, y_score))
    eers.append(compute_eer(y_test, y_score))

    # print(f"Accuracy: {accs[-1]}")
    # print(f"Precision: {precisions[-1]}")
    # print(f"Recall: {recalls[-1]}")
    # print(f"F1: {f1s[-1]}")
    # print(f"EER: {eers[-1]}")
    # print()


print("===== 5 Fold CV Result =====")
print("Accuracy :", np.mean(accs))
print("Precision:", np.mean(precisions))
print("Recall   :", np.mean(recalls))
print("F1-score :", np.mean(f1s))
print("ROC-AUC  :", np.mean(aucs))
print("EER      :", np.mean(eers))

spoof: 8931
bonafide: 4861
Training phase: 1
Training phase: 2
Training phase: 3
Training phase: 4
Training phase: 5
===== 5 Fold CV Result =====
Accuracy : 0.7728397615451154
Precision: 0.7895066847566156
Recall   : 0.4850866396829626
F1-score : 0.6008462089819446
ROC-AUC  : 0.8757800872612578
EER      : 0.20199224083855594
